# Configure Fabric Workspace

This notebook configures a Microsoft Fabric workspace for the benchmarking solution:

1. **Provision workspace identity** — creates a managed service principal for the workspace
2. **Grant contributor access** — assigns the workspace identity the Contributor role
3. **Create shared cloud connection** — creates a "Fabric Data Pipelines (Workspace Identity)" connection
4. **Update variable library** — writes the notebook GUID into the variable library for pipeline orchestration

This notebook runs **locally** (outside Fabric) using `azure-identity` for interactive browser authentication.
Set the `WORKSPACE_NAME` variable in the next cell before running.

In [5]:
import logging

from fabric_admin import (
    FabricConnections,
    FabricRestClient,
    FabricVariableLibrary,
    FabricWorkspace,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s")

In [13]:
# === CONFIGURATION ===
# Set the workspace name to configure
WORKSPACE_NAME = "fabric_performance_benchmark_workspace"

# Connection settings
CONNECTION_DISPLAY_NAME = "Fabric Data Pipelines (Workspace Identity)"

# Connection type — discovered via the supportedConnectionTypes API
CONNECTION_TYPE = "FabricDataPipelines"
CREATION_METHOD = "FabricDataPipelines.Actions"
CONNECTION_PARAMETERS: list[dict] = []

# Variable library settings (for notebook GUID update)
VARIABLE_LIBRARY_NAME = "benchmark_1_variables"
NOTEBOOK_NAME = "pyspark_benchmark"

## Authenticate and resolve workspace

In [7]:
# Authenticate via interactive browser login and resolve the workspace
client = FabricRestClient()
workspace = FabricWorkspace(client, WORKSPACE_NAME)

2026-04-14 08:45:23,683 fabric_admin._client INFO No token supplied — acquiring token via interactive browser login
2026-04-14 08:45:23,685 azure.core.pipeline.policies.http_logging_policy INFO Request URL: 'https://login.microsoftonline.com/organizations/v2.0/.well-known/openid-configuration'
Request method: 'GET'
Request headers:
    'User-Agent': 'azsdk-python-identity/1.25.1 Python/3.13.12 (Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39)'
No body was attached to the request
2026-04-14 08:45:23,876 azure.core.pipeline.policies.http_logging_policy INFO Response status: 200
Response headers:
    'Cache-Control': 'max-age=86400, private'
    'Content-Type': 'application/json; charset=utf-8'
    'Strict-Transport-Security': 'REDACTED'
    'X-Content-Type-Options': 'REDACTED'
    'Access-Control-Allow-Origin': 'REDACTED'
    'Access-Control-Allow-Methods': 'REDACTED'
    'P3P': 'REDACTED'
    'x-ms-request-id': '15a0cba3-dc0b-444c-b78f-821fabbe0000'
    'x-ms-ests-server': 

In [8]:
workspace_id = workspace.workspace_id

2026-04-14 08:45:28,023 fabric_admin.workspace INFO Resolving workspace ID for 'fabric_performance_benchmark_workspace'
2026-04-14 08:45:28,315 fabric_admin.workspace INFO Resolved workspace 'fabric_performance_benchmark_workspace' → 23d2362b-6b5f-4894-8472-c09991fc07a6


## Step 1 & 2: Provision workspace identity with contributor access

This provisions a workspace identity (managed service principal) and assigns it the
Contributor role. Both operations are idempotent — safe to re-run.

In [9]:
identity = workspace.ensure_identity_with_contributor_access()

2026-04-14 08:45:28,321 fabric_admin.workspace INFO Provisioning workspace identity for workspace 23d2362b-6b5f-4894-8472-c09991fc07a6
2026-04-14 08:45:29,097 fabric_admin.workspace INFO Workspace identity already exists (HTTP 400)
2026-04-14 08:45:29,099 fabric_admin.workspace WARNING Workspace identity already existed — cannot determine service principal ID. Verify contributor access manually in workspace settings.


## Step 3: Discover connection types and create shared cloud connection

First, discover the exact connection type string for Fabric Data Pipelines,
then create the connection.

In [12]:
# Discover pipeline-related connection types that support WorkspaceIdentity
connections = FabricConnections(client)
candidates = connections.find_connection_type(
    keywords=["pipeline", "datafactory", "fabricdatapipeline", "datapipeline"],
    credential_type="WorkspaceIdentity",
)

# Print results concisely so we can pick the right type/method
for ct in candidates:
    for method in ct.get("creationMethods", []):
        print(f"  type={ct['type']!r}, creationMethod={method['name']!r}, params={method.get('parameters', [])}")

if not candidates:
    # Broader search — show ALL types supporting WorkspaceIdentity
    all_types = connections.list_supported_types()
    print("\nAll types supporting WorkspaceIdentity:")
    for ct in all_types:
        if "WorkspaceIdentity" in ct.get("supportedCredentialTypes", []):
            for method in ct.get("creationMethods", []):
                print(f"  type={ct['type']!r}, creationMethod={method['name']!r}, params={method.get('parameters', [])}")

2026-04-14 08:47:55,401 fabric_admin.connections INFO Fetching supported connection types
2026-04-14 08:47:55,892 fabric_admin.connections INFO Found 264 supported connection types
2026-04-14 08:47:55,893 fabric_admin.connections INFO Matched connection type 'AzureDataFactory' (credentials: ['OAuth2', 'ServicePrincipal', 'WorkspaceIdentity'])
2026-04-14 08:47:55,894 fabric_admin.connections INFO   Creation method 'AzureDataFactory.Actions', parameters: [
  {
    "name": "subscriptionId",
    "dataType": "Text",
    "required": true,
    "allowedValues": null
  },
  {
    "name": "resourceGroup",
    "dataType": "Text",
    "required": true,
    "allowedValues": null
  },
  {
    "name": "dataFactoryName",
    "dataType": "Text",
    "required": true,
    "allowedValues": null
  }
]
2026-04-14 08:47:55,894 fabric_admin.connections INFO Matched connection type 'FabricDataPipelines' (credentials: ['OAuth2', 'ServicePrincipal', 'WorkspaceIdentity'])
2026-04-14 08:47:55,894 fabric_admin.con

  type='AzureDataFactory', creationMethod='AzureDataFactory.Actions', params=[{'name': 'subscriptionId', 'dataType': 'Text', 'required': True, 'allowedValues': None}, {'name': 'resourceGroup', 'dataType': 'Text', 'required': True, 'allowedValues': None}, {'name': 'dataFactoryName', 'dataType': 'Text', 'required': True, 'allowedValues': None}]
  type='FabricDataPipelines', creationMethod='FabricDataPipelines.Actions', params=[]


In [14]:
# Create (or find existing) shared cloud connection
connection_id = connections.create_cloud_connection(
    display_name=CONNECTION_DISPLAY_NAME,
    connection_type=CONNECTION_TYPE,
    creation_method=CREATION_METHOD,
    parameters=CONNECTION_PARAMETERS,
)

2026-04-14 08:48:16,324 fabric_admin.connections INFO Creating cloud connection 'Fabric Data Pipelines (Workspace Identity)' (type=FabricDataPipelines, credential=WorkspaceIdentity)
2026-04-14 08:48:16,988 fabric_admin.connections INFO Connection 'Fabric Data Pipelines (Workspace Identity)' already exists — looking up existing GUID
2026-04-14 08:48:17,450 fabric_admin.connections INFO Found existing connection 'Fabric Data Pipelines (Workspace Identity)' → 24ea66da-17ae-45cf-b4db-1151b2effebc


## Step 4 (Optional): Update variable library with notebook GUID

Looks up a notebook by name in the workspace and writes its GUID into the
variable library so that pipelines can reference it.

In [15]:
# Look up the notebook GUID
notebook_id = workspace.get_item_id(NOTEBOOK_NAME, item_type="Notebook")

2026-04-14 08:49:11,292 fabric_admin.workspace INFO Looking up Notebook 'pyspark_benchmark' in workspace 23d2362b-6b5f-4894-8472-c09991fc07a6
2026-04-14 08:49:11,731 fabric_admin.workspace INFO Found Notebook 'pyspark_benchmark' → 3a4f55e4-c646-42aa-825c-e8b4a44524dd


In [16]:
# Update the variable library
var_lib = FabricVariableLibrary(client, workspace_id)
library_id = var_lib.find_library(VARIABLE_LIBRARY_NAME)
var_lib.update_variable(library_id, f"{NOTEBOOK_NAME}_notebook_id", notebook_id)

2026-04-14 08:49:14,865 fabric_admin.variable_library INFO Looking up variable library 'benchmark_1_variables'
2026-04-14 08:49:15,091 fabric_admin.variable_library INFO Found variable library 'benchmark_1_variables' → d088e269-3485-41af-8bfd-6e9e99c6f79a
2026-04-14 08:49:15,093 fabric_admin.variable_library INFO Fetching definition for variable library d088e269-3485-41af-8bfd-6e9e99c6f79a
2026-04-14 08:49:37,687 fabric_admin.variable_library INFO Decoded variables.json — 8 variables found
2026-04-14 08:49:37,689 fabric_admin.variable_library INFO Variable 'pyspark_benchmark_notebook_id' updated: '678955e4-c646-42aa-825c-e8b4a44524dd' → '3a4f55e4-c646-42aa-825c-e8b4a44524dd'
2026-04-14 08:49:37,690 fabric_admin.variable_library INFO Updating definition for variable library d088e269-3485-41af-8bfd-6e9e99c6f79a
2026-04-14 08:49:59,523 fabric_admin.variable_library INFO Variable library updated successfully — 'pyspark_benchmark_notebook_id' = '3a4f55e4-c646-42aa-825c-e8b4a44524dd'
